# runtime

> Execute code and edit values in the live kernel namespace

In [ ]:
#| default_exp runtime

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
import paar.runtime as R
from paar.runtime import run, ExecResult

class _Shell:
    "Minimal IPython stand-in whose run_cell execs into a shared user_ns."
    def __init__(self): self.user_ns = {}
    def run_cell(self, code, store_history=True):
        import types
        ns = self.user_ns
        err = None; result = None
        try:
            block = compile(code, '<t>', 'exec')
            exec(block, ns)
            # emulate IPython binding `_` for a trailing expression
            import ast
            body = ast.parse(code).body
            if body and isinstance(body[-1], ast.Expr):
                result = eval(compile(ast.Expression(body[-1].value), '<t>', 'eval'), ns)
                ns['_'] = result
        except Exception as e:
            err = e
        return types.SimpleNamespace(result=result, error_in_exec=err,
                                     error_before_exec=None, success=err is None)

def test_run_global_assignment_mutates_ns():
    sh = _Shell(); R.get_ipython = lambda: sh
    r = run('x = 5')
    assert isinstance(r, ExecResult) and r.ok is True
    assert r.result is None and sh.user_ns['x'] == 5   # assignment: no result row
def test_run_global_expression_makes_result_row():
    sh = _Shell(); R.get_ipython = lambda: sh
    r = run('3 + 4')
    assert r.ok is True and r.result is not None
    assert r.result.name == 'result' and r.result.value == '7' and r.result.accessor == ('_',)
def test_run_global_modify_existing():
    sh = _Shell(); sh.user_ns['n'] = 1; R.get_ipython = lambda: sh
    run('n = n + 41'); assert sh.user_ns['n'] == 42
def test_run_error_is_captured():
    sh = _Shell(); R.get_ipython = lambda: sh
    r = run('1/0')
    assert r.ok is False and 'ZeroDivisionError' in r.error
def test_run_no_ipython():
    R.get_ipython = lambda: None
    r = run('x=1'); assert r.ok is False and 'no IPython' in r.error
for t in [test_run_global_assignment_mutates_ns, test_run_global_expression_makes_result_row,
          test_run_global_modify_existing, test_run_error_is_captured, test_run_no_ipython]: t()

In [ ]:
#| export
from __future__ import annotations
from dataclasses import dataclass
from paar.core import VarInfo
from paar.snapshot import _var_info
from paar.providers import value_str
from IPython.utils.capture import capture_output
try: from IPython import get_ipython
except Exception:
    def get_ipython(): return None

@dataclass
class ExecResult:
    "Outcome of running code: last value as an inspector row, captured stdout, and error text."
    ok: bool
    result: VarInfo|None = None
    stdout: str = ''
    error: str|None = None

def _fmt_err(res):
    "Formatted exception text from a run_cell result, or None."
    e = getattr(res, 'error_in_exec', None) or getattr(res, 'error_before_exec', None)
    return f'{type(e).__name__}: {e}' if e is not None else None

def run(code, scope='global'):
    "Run `code` in the kernel; scope='global' mutates user_ns, 'isolated' uses a scratch copy."
    ip = get_ipython()
    if ip is None: return ExecResult(ok=False, error='no IPython kernel')
    with capture_output() as cap:
        res = ip.run_cell(code, store_history=True)
    err = _fmt_err(res)
    result = _var_info('result', res.result, ('_',), '_') if res.result is not None else None
    return ExecResult(ok=bool(res.success) and err is None, result=result, stdout=cap.stdout, error=err)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()